World Bank - Country Lookup Table
=================================

In [ ]:
from pathlib import Path

import pandas as pd
import wbgapi as wb

from utils.paths import STG_WORLD_BANK_DIR

In [ ]:
WB_REGIONS_FILENAME = 'stg_world_bank__regions.csv'
WB_REGIONS_FILEPATH = STG_WORLD_BANK_DIR / WB_REGIONS_FILENAME

In [ ]:
wb_economies: pd.DataFrame = wb.economy.DataFrame()

In [ ]:
print(wb_economies.info())
wb_economies.head(1)

In [ ]:
for col in wb_economies.columns:
    print(col)

In [ ]:
wb_economies_df = (
    wb_economies
    .reset_index()
    .rename(columns={
        'id': 'wb_economy_code',
        'name': 'wb_economy_name',
        'region': 'region_code',
        'adminregion': 'admin_region_code'
    })
    .replace({'': pd.NA})
)

In [ ]:
for col in wb_economies_df.columns:
    print(col)

In [ ]:
wb_regions: pd.Series = wb.region.Series()  # region API call

In [ ]:
# Create map of all World Bank economies to their
# respective regional groups (i.e. 'regions') 

wb_regions_df = (
    wb_regions
    .to_frame(name='region_name')
    .reset_index()
    .rename(columns={'index': 'region_code'})
)

wb_region_map_df = (
    wb_economies_df
    .loc[:, ['wb_economy_code', 'wb_economy_name', 'region_code']]
    .merge(
        wb_regions_df,
        on='region_code',
        how='inner'
    )
    .loc[:, [
        'region_code',
        'region_name',
        'wb_economy_code',
        'wb_economy_name'
    ]]
    .sort_values(by=['region_code', 'wb_economy_code'], ignore_index=True)
    .reset_index(drop=True)
)

In [ ]:
wb_region_map_df

In [ ]:
wb_admin_region_map_df = (
    wb_economies_df
    .loc[:, ['wb_economy_code', 'wb_economy_name', 'admin_region_code']]
    .merge(
        wb_regions_df,
        left_on='admin_region_code',
        right_on='region_code',
        how='inner'
    )
    .rename(columns={
        'region_name': 'admin_region_name'
    })
    .loc[:, [
        'admin_region_code',
        'admin_region_name',
        'wb_economy_code',
        'wb_economy_name'
    ]]
)

In [ ]:
wb_admin_region_map_df

In [ ]:
# TODO: Execute code below to join admin region name with wb_economies_df

wb_combined_df = (
    wb_economies_df
    .merge(
        wb_admin_region_map_df[['admin_region_code', 'admin_region_name']],
        left_on='admin_region',
        right_on='admin_region_code',
        how='left'
    )
    .drop(columns='admin_region')
) 

In [ ]:
wb_combined_df

In [ ]:
# Replace wb_economies_df if join w/ admin regions works as desired
wb_economies_df = wb_combined_df.copy()

In [ ]:
# Check that filename ends in '.csv', then write df to CSV
if WB_REGIONS_FILENAME.endswith('.csv'):
    wb_economies_df.to_csv(wb_regions_path, index=False)
    
    relative_dir = common_tables_dir.relative_to(project_dir)
    
    print(f"DataFrame written to file '{WB_REGIONS_FILENAME}'")
    print(f"Filepath: .\\{relative_dir}\\{WB_REGIONS_FILENAME}")
else:
    print("Abort write: filename missing '.csv' file extension")